[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/python-ai-business-data-science/blob/main/02_data_science/09_matplotlib_basics.ipynb)

# 📓 Notebook 9 — Matplotlib Basics: Communicating with Charts

> **Module:** Data Science Libraries · **Estimated time:** 40–55 min · **Difficulty:** Beginner

A plot answers questions that tables cannot. Distributions, trends, outliers, relationships — they are *seen* before they are *understood*. Matplotlib is the foundational plotting library of Python: pandas' built-in `.plot`, seaborn, scikit-learn's diagnostic plots, even plotly's static export — they all sit on top of it.

In this notebook every example is anchored in **AI support-operations data**: cost-per-call, latency distributions, monthly automation rates, model A/B comparisons. The exercises and bonus project build directly toward the **2×2 executive dashboard** used in the NB 24 capstone — by the end you will have all the pieces.

## 🎯 Learning objectives

1. Use the **Figure / Axes** object-oriented API confidently.
2. Build line, bar, scatter, histogram, and box plots.
3. Add titles, axis labels, legends, gridlines, and annotations.
4. Compose **multi-panel layouts** with `plt.subplots(rows, cols)`.
5. Apply a consistent **palette** and style across a figure.
6. Save publication-quality figures to PNG/PDF.

## ✅ Prerequisites

Notebooks 1–8. (Most cells use NumPy; a few use pandas at the end.)

## 1. Setup — the canonical imports

Just three lines, and they are the same in every notebook you'll ever see.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A consistent, professional look. Run once at the top of any notebook.
plt.rcParams.update({
    "figure.figsize"   : (8, 5),
    "figure.dpi"       : 100,
    "axes.grid"        : True,
    "grid.alpha"       : 0.3,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.size"        : 11,
})

print(f"matplotlib version: {plt.matplotlib.__version__}")


> 💡 **rcParams.** Setting these at the top of a notebook gives every figure a consistent look. Tweaking the default style is the single biggest *return-per-line* investment you can make in your plotting code.

## 2. The Figure / Axes mental model

Matplotlib has two layers you need to keep straight:

```
┌────────────── Figure (the whole window) ──────────────┐
│                                                        │
│   ┌────── Axes (one plot inside) ──────┐               │
│   │                                    │               │
│   │   data lives here                  │               │
│   │                                    │               │
│   └────────────────────────────────────┘               │
│                                                        │
└────────────────────────────────────────────────────────┘
```

A **Figure** is the canvas; an **Axes** is one plot inside it. You can have many Axes in one Figure. We always use the object-oriented form `fig, ax = plt.subplots(...)` — it scales smoothly from "one plot" to "twelve-panel dashboard".

## 3. Your first line plot — automation rate over a year

We'll use the same support-operations theme as NB7 throughout. First, a single line plot of monthly automation rate.

In [ ]:
months = np.arange(1, 13)
auto_rate = np.array([0.55, 0.58, 0.60, 0.63, 0.65, 0.68, 0.70, 0.72, 0.74, 0.76, 0.78, 0.81])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(months, auto_rate, color="#4C72B0", linewidth=2, marker="o")
ax.set_title("Monthly automation rate — Chat channel")
ax.set_xlabel("Month")
ax.set_ylabel("Automation rate")
ax.set_xticks(months)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


The seven lines above are essentially every plot you'll ever make:

1. `fig, ax = plt.subplots(...)` — create a Figure and one Axes.
2. `ax.plot(...)` — draw the data.
3. `ax.set_title / set_xlabel / set_ylabel` — label it.
4. `ax.set_xticks / set_ylim / ...` — adjust axes when needed.
5. `plt.tight_layout()` — fix overlapping labels.
6. `plt.show()` — render.

Everything else is variations on this recipe.

## 4. Multiple lines, legends, line styles

Same x-axis, multiple lines — perfect for "the same metric across categories".

In [ ]:
months = np.arange(1, 13)
chat     = np.array([0.55, 0.58, 0.60, 0.63, 0.65, 0.68, 0.70, 0.72, 0.74, 0.76, 0.78, 0.81])
email    = np.array([0.42, 0.45, 0.47, 0.50, 0.52, 0.55, 0.57, 0.60, 0.62, 0.65, 0.67, 0.70])
phone    = np.array([0.15, 0.16, 0.17, 0.18, 0.20, 0.21, 0.22, 0.23, 0.24, 0.25, 0.26, 0.28])
web_form = np.array([0.62, 0.65, 0.68, 0.70, 0.72, 0.74, 0.76, 0.78, 0.80, 0.82, 0.83, 0.85])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(months, chat,     label="Chat",     color="#55A467", linewidth=2, marker="o")
ax.plot(months, email,    label="Email",    color="#4C72B0", linewidth=2, marker="s")
ax.plot(months, phone,    label="Phone",    color="#C44E52", linewidth=2, marker="^", linestyle="--")
ax.plot(months, web_form, label="Web Form", color="#DD8452", linewidth=2, marker="d")

ax.set_title("Automation rate by channel (one year)")
ax.set_xlabel("Month")
ax.set_ylabel("Automation rate")
ax.set_xticks(months)
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


> 💡 **Common line styles:** `"-"` solid, `"--"` dashed, `":"` dotted, `"-."` dash-dot. **Markers:** `"o"` circle, `"s"` square, `"^"` triangle, `"d"` diamond, `"x"` cross, `"."` point.

For colour, use hex codes (`"#4C72B0"`), CSS names (`"crimson"`), or any standard matplotlib colour. **Stick to a small, consistent palette** — your audience will read the chart faster.

## 5. Scatter plots — relationships between two variables

For *"is X related to Y?"* use a scatter. We'll plot cost vs satisfaction for a batch of calls, coloured by the customer segment.

In [ ]:
rng = np.random.default_rng(42)
n = 80

# Mock: cost driven by tokens, satisfaction mildly inversely related to cost
cost          = rng.uniform(0.001, 0.012, size=n)
satisfaction  = 4.5 - 80 * cost + rng.normal(0, 0.25, size=n)
segment_idx   = rng.integers(0, 3, size=n)
segment_names = np.array(["SMB", "Mid-market", "Enterprise"])
colours       = np.array(["#4C72B0", "#DD8452", "#55A467"])

fig, ax = plt.subplots(figsize=(8, 5))

# One scatter per segment — keeps the legend tidy
for i, name in enumerate(segment_names):
    sel = segment_idx == i
    ax.scatter(cost[sel], satisfaction[sel],
               c=colours[i], s=55, edgecolor="black", alpha=0.8, label=name)

# A simple linear fit on top — communicates the trend
m, b = np.polyfit(cost, satisfaction, 1)
xs = np.linspace(cost.min(), cost.max(), 50)
ax.plot(xs, m*xs + b, "k--", linewidth=1.4, label=f"fit: slope={m:+.1f}")

ax.set_title("Cost per call vs customer satisfaction")
ax.set_xlabel("Cost per call (USD)")
ax.set_ylabel("Satisfaction (1–5)")
ax.legend()
plt.tight_layout()
plt.show()


**Reading the chart.** The downward slope of the trend line says higher-cost calls tend to come with slightly lower satisfaction — possibly because expensive calls are the hard ones the bot couldn't fully handle. The colour encoding lets you check whether *segment* explains anything beyond cost (it mostly doesn't here — the segments overlap heavily).

## 6. Bar charts — comparing categories

In [ ]:
channels = ["Email", "Chat", "Phone", "Web Form", "Social"]
annual_cost_usd = [9_400, 14_200, 11_800, 6_700, 3_900]
palette = ["#4C72B0", "#55A467", "#C44E52", "#DD8452", "#8172B2"]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(channels, annual_cost_usd, color=palette, edgecolor="black")

# Annotate each bar with its value — never make people read off the y-axis
for bar, val in zip(bars, annual_cost_usd):
    ax.text(bar.get_x() + bar.get_width()/2, val, f"${val:,}",
            ha="center", va="bottom", fontsize=10)

ax.set_title("Annual support-channel spend")
ax.set_ylabel("Spend (USD)")
ax.set_ylim(0, max(annual_cost_usd) * 1.15)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### Horizontal bar charts

Use `barh` whenever the category labels are long — they read more comfortably horizontally.

In [ ]:
skills = ["Python", "Pandas", "SQL", "Statistics", "Machine Learning",
          "Prompt engineering", "Cloud ops", "Communication"]
importance = [95, 90, 88, 92, 85, 87, 78, 90]

order = np.argsort(importance)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(np.array(skills)[order], np.array(importance)[order],
        color="#4C72B0", edgecolor="black")
ax.set_title("Most useful skills for a modern AI-data role (survey)")
ax.set_xlabel("Self-reported importance (0–100)")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()


## 7. Histograms — distributions

A histogram tells you the *shape* of one variable: is it bell-shaped, skewed, bimodal? Crucial before you start modelling.

In [ ]:
rng = np.random.default_rng(0)

# Two simulated latency distributions
fast_model = rng.normal(loc=1800, scale=400,  size=400)
slow_model = rng.normal(loc=2700, scale=700,  size=400)
fast_model = np.clip(fast_model, 200, None)
slow_model = np.clip(slow_model, 200, None)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(fast_model, bins=25, alpha=0.65, label="Model A (fast)",
        color="#4C72B0", edgecolor="black")
ax.hist(slow_model, bins=25, alpha=0.65, label="Model B (slow)",
        color="#DD8452", edgecolor="black")

ax.axvline(fast_model.mean(), color="#4C72B0", linestyle="--", linewidth=1.5)
ax.axvline(slow_model.mean(), color="#DD8452", linestyle="--", linewidth=1.5)

ax.set_title("Latency distribution — two model candidates")
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Number of calls")
ax.legend()
plt.tight_layout()
plt.show()


**Reading a histogram.** The x-axis is the value, the y-axis the count of observations in each bin. Two overlapping histograms with `alpha ≈ 0.65` is a clean way to compare two distributions. Vertical dashed lines highlight the means — much more informative than a single number.

## 8. Box plots — distribution at a glance

When you have *several* distributions to compare side by side, a box plot is denser than a row of histograms.

In [ ]:
rng = np.random.default_rng(1)
models = ["gpt-4o-mini", "claude-haiku", "internal-llm", "open-source-q4"]
samples = [
    rng.normal(1800, 400, 60),
    rng.normal(2400, 600, 60),
    rng.normal(1500, 350, 60),
    rng.normal(2100, 800, 60),
]
samples = [np.clip(s, 200, None) for s in samples]

fig, ax = plt.subplots(figsize=(8, 5))
box = ax.boxplot(samples, tick_labels=models, patch_artist=True,
                 medianprops=dict(color="black", linewidth=2))

palette = ["#4C72B0", "#DD8452", "#55A467", "#8172B2"]
for patch, c in zip(box["boxes"], palette):
    patch.set_facecolor(c); patch.set_alpha(0.75)

ax.set_title("Latency distribution per candidate model")
ax.set_ylabel("Latency (ms)")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()


**Reading a box plot.** The box covers the interquartile range (Q1 to Q3). The line inside is the **median**. Whiskers extend to data within 1.5×IQR; points beyond are outliers (drawn as dots). A wide box = high variance, a tight box = consistent performance.

## 9. Multi-panel layouts — `subplots(rows, cols)`

This is the building block of every dashboard you'll ever make.

In [ ]:
rng = np.random.default_rng(7)

# Pretend we have 300 calls and three "channels" of information about them
n = 300
cost       = rng.gamma(2.0, 0.001, size=n)         # right-skewed
satisfaction = rng.normal(4.0, 0.5, size=n).clip(1, 5)
day        = np.arange(n)
volume     = np.cumsum(rng.normal(0, 1, n)) + 50    # cumulative trend

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
fig.suptitle("Mini AI-ops dashboard", fontsize=15, fontweight="bold")

# (0, 0) Histogram of cost per call
axes[0, 0].hist(cost * 100, bins=25, color="#4C72B0", edgecolor="black")
axes[0, 0].set_title("Cost per call (¢)")
axes[0, 0].set_xlabel("Cost (cents)")
axes[0, 0].set_ylabel("Number of calls")

# (0, 1) Scatter of cost vs satisfaction
axes[0, 1].scatter(cost * 100, satisfaction, alpha=0.4, color="#DD8452",
                   edgecolor="black", s=20)
axes[0, 1].set_title("Cost vs satisfaction")
axes[0, 1].set_xlabel("Cost (cents)")
axes[0, 1].set_ylabel("Satisfaction (1–5)")

# (1, 0) Line trend
axes[1, 0].plot(day, volume, color="#55A467", linewidth=1.6)
axes[1, 0].set_title("Daily volume index (cumulative)")
axes[1, 0].set_xlabel("Day of year")
axes[1, 0].set_ylabel("Volume index")

# (1, 1) Box plot of satisfaction by quarter
quarters = ["Q1", "Q2", "Q3", "Q4"]
groups = [satisfaction[(day >= i*75) & (day < (i+1)*75)] for i in range(4)]
axes[1, 1].boxplot(groups, tick_labels=quarters, patch_artist=True)
axes[1, 1].set_title("Satisfaction by quarter")
axes[1, 1].set_ylabel("Satisfaction (1–5)")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


> 💡 `axes` is a NumPy array of Axes objects — `axes[0, 0]` is top-left, `axes[1, 1]` is bottom-right. For a single row or column you get a 1-D array (`axes[0]`, `axes[1]`).

This 2×2 layout is the **canonical "executive dashboard"** shape — and it's exactly the layout you'll use in the NB 24 capstone. Get comfortable with it.

## 10. Annotations — making the punchline visible

A *good* chart leaves no ambiguity about what the reader should look at.

In [ ]:
months = np.arange(1, 25)
auto_rate = np.array([
    0.40, 0.42, 0.45, 0.46, 0.48, 0.50, 0.51, 0.52, 0.53, 0.55, 0.56, 0.58,
    0.60, 0.62, 0.65, 0.55,    # bug regression in month 16
    0.66, 0.69, 0.71, 0.73, 0.74, 0.76, 0.77, 0.79,
])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(months, auto_rate, marker="o", color="#4C72B0", linewidth=2)

ax.annotate("Regression\n(post-deploy bug)",
            xy=(16, 0.55), xytext=(13, 0.42),
            arrowprops=dict(arrowstyle="->", color="black"),
            fontsize=10)

ax.annotate("Quarterly review",
            xy=(24, 0.79), xytext=(20, 0.66),
            arrowprops=dict(arrowstyle="->", color="black"),
            fontsize=10)

ax.set_title("Automation rate over two years")
ax.set_xlabel("Month")
ax.set_ylabel("Automation rate")
ax.set_ylim(0.3, 0.9)
plt.tight_layout()
plt.show()


## 11. Heatmaps — visualising matrices

Heatmaps are perfect for correlation matrices, confusion matrices, or any "every-row-by-every-column" comparison.

In [ ]:
# A small confusion-matrix-ish view: predicted vs actual sentiment
labels = ["positive", "neutral", "negative"]
cm = np.array([
    [42, 3,  1],     # true positive
    [ 4, 38, 5],     # true neutral
    [ 1, 6, 35],     # true negative
])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(3), labels)
ax.set_yticks(range(3), labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Sentiment-classifier confusion matrix")

# Annotate every cell
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j],
                ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black",
                fontsize=12)

fig.colorbar(im, ax=ax, label="Number of examples")
plt.tight_layout()
plt.show()


**This exact plot is what you'll use in NB 14** to evaluate a real classifier. The diagonal shows correct predictions; off-diagonal cells show where the model is confused.

## 12. Saving figures

Drop `plt.savefig("name.png", dpi=200, bbox_inches="tight")` *before* `plt.show()` to save your figure to disk. Common formats:

| Extension | Use for                                              |
|-----------|------------------------------------------------------|
| `.png`    | reports, web, default raster                          |
| `.pdf`    | papers, slides, infinitely scalable vector            |
| `.svg`    | web, when you may edit later in Illustrator/Inkscape  |

```python
fig, ax = plt.subplots()
ax.plot(x, y)
plt.savefig("automation_rate.png", dpi=200, bbox_inches="tight")
plt.show()
```

## 13. Common pitfalls

| Pitfall                                       | Symptom                                | Fix |
|-----------------------------------------------|----------------------------------------|-----|
| Squished figure / overlapping labels          | crowded titles                         | call `plt.tight_layout()` before `plt.show()` |
| Multiple plots in one cell mixed by accident  | extra empty axes                       | always open a new figure with `plt.subplots(...)` |
| No axis labels / units                        | viewer can't tell what is plotted      | always set title + xlabel + ylabel |
| 3-D pie charts and other chartjunk            | takes longer to read                   | almost never use them; bar > pie > 3-D pie |
| Hard-to-distinguish colours                   | viewers can't tell categories apart    | use a small consistent palette; consider colourblind-safe colormaps |

## 🧪 Practice exercises

### Exercise 1 — ⭐ Three lines, one plot

On the interval `x ∈ [0, 5]`, plot `y₁ = x`, `y₂ = x²`, `y₃ = 2ˣ` on the same axes. Add labels, a legend, a title, and a grid.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
x = np.linspace(0, 5, 200)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, x,     label="y = x",   linewidth=2)
ax.plot(x, x**2,  label="y = x²",  linewidth=2)
ax.plot(x, 2**x,  label="y = 2ˣ",  linewidth=2)

ax.set_title("Linear vs quadratic vs exponential growth")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()
```

A useful intuition for AI cost models: exponential growth (e.g., context length × cost) overtakes anything else *eventually* — visible immediately from a chart like this.
</details>

### Exercise 2 — ⭐⭐ Histogram with mean line

Generate 1,000 simulated latencies from a normal distribution with mean 2000 ms and std 400 ms. Plot a histogram, overlay vertical dashed lines for the mean and (mean ± std).

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
rng = np.random.default_rng(0)
data = rng.normal(2000, 400, 1000)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(data, bins=30, color="#4C72B0", edgecolor="black", alpha=0.8)

m, s = data.mean(), data.std()
ax.axvline(m,     color="red",    linestyle="--", label=f"mean = {m:.0f}")
ax.axvline(m - s, color="orange", linestyle=":")
ax.axvline(m + s, color="orange", linestyle=":", label=f"±1 std = {s:.0f}")

ax.set_title("Simulated latency distribution")
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Number of calls")
ax.legend()
plt.tight_layout()
plt.show()
```
</details>

### Exercise 3 — ⭐⭐ Subplots side-by-side

Create a figure with **two side-by-side subplots**:

1. Left: a bar chart of monthly spend by channel (sample data below).
2. Right: a pie chart of the same data.

Use the same colour palette for both, and add `fig.suptitle(...)`.

In [ ]:
# Your code here  👇
channels = ["Email", "Chat", "Phone", "Web Form"]
spend    = [12_200, 9_500, 14_500, 8_800]


<details>
<summary>💡 <b>Solution</b></summary>

```python
palette = ["#4C72B0", "#DD8452", "#55A467", "#C44E52"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle("Monthly support-channel spend", fontsize=14, fontweight="bold")

# Bar
axes[0].bar(channels, spend, color=palette, edgecolor="black")
axes[0].set_title("Bar")
axes[0].set_ylabel("Spend (USD)")
axes[0].grid(axis="y", alpha=0.3)

# Pie
axes[1].pie(spend, labels=channels, colors=palette,
            autopct="%1.0f%%", startangle=90, wedgeprops=dict(edgecolor="white"))
axes[1].set_title("Pie")

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()
```

A pie chart is *only* a good choice when you have ≤ 5 categories and absolute proportions are the point. For ranking categories, a bar chart wins every time.
</details>

### Exercise 4 — ⭐⭐ Annotated scatter

Generate 100 random (cost, satisfaction) points where `satisfaction = 4.5 − 100·cost + noise`. Plot them, add the regression line, and annotate the **largest residual** (the point furthest from the line).

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
rng = np.random.default_rng(3)
cost = rng.uniform(0.001, 0.020, 100)
sat  = 4.5 - 100 * cost + rng.normal(0, 0.3, 100)

m, b = np.polyfit(cost, sat, 1)
xs   = np.linspace(cost.min(), cost.max(), 50)
pred = m * cost + b
res  = sat - pred
i    = np.argmax(np.abs(res))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(cost, sat, alpha=0.6, color="#4C72B0", s=30, edgecolor="black")
ax.plot(xs, m*xs + b, "r--", linewidth=1.5, label=f"fit: slope={m:.0f}")

ax.scatter(cost[i], sat[i], color="red", s=120, edgecolor="black", zorder=3)
ax.annotate(f"largest residual\n(res = {res[i]:+.2f})",
            xy=(cost[i], sat[i]),
            xytext=(cost[i] + 0.002, sat[i] - 0.5),
            arrowprops=dict(arrowstyle="->", color="black"),
            fontsize=10)

ax.set_title("Cost vs satisfaction with largest outlier marked")
ax.set_xlabel("Cost per call (USD)")
ax.set_ylabel("Satisfaction (1–5)")
ax.legend()
plt.tight_layout()
plt.show()
```

Highlighting an outlier directly on the chart is one of the highest-leverage things you can do with a plot — it makes the next question (*"why?"*) jump out.
</details>

### Exercise 5 — ⭐⭐ Debug me 🐞

The plot below should show *two* sine curves with different phases. It currently only shows one and has no axis labels. Fix it.

In [ ]:
# 👇 Your fixed/corrected version goes here — write or paste it below.
x = np.linspace(0, 2*np.pi, 200)

plt.figure()
plt.plot(x, np.sin(x))
plt.plot(x, np.sin(x))       # bug: same curve again
plt.title("Two sine curves")
plt.show()


<details>
<summary>💡 <b>Solution</b></summary>

Two issues: both lines plot the same `sin(x)`, and the axes are unlabelled.

```python
x = np.linspace(0, 2*np.pi, 200)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, np.sin(x),         label="sin(x)")
ax.plot(x, np.sin(x + np.pi/4), label="sin(x + π/4)")
ax.set_title("Two sine curves with a phase shift")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()
```

A useful habit: never trust a chart you just drew. Look at it, ask "would a stranger understand this in 5 seconds?", and fix anything that fails the test.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise A — ⭐⭐⭐ Annotated line chart with markers

Plot a line chart of `y = [3, 5, 4, 8, 12, 7, 9]` against `x = [1..7]`. Annotate the maximum with an arrow and the text 'peak'. Add axis labels, title, gridlines.


<details>
<summary>💡 <b>Solution</b></summary>

```python
x = np.arange(1, 8)
y = np.array([3, 5, 4, 8, 12, 7, 9])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, y, marker="o", lw=2, color="#4C72B0")
ax.set_title("Demo annotated line chart")
ax.set_xlabel("step"); ax.set_ylabel("value")

i = int(np.argmax(y))
ax.annotate("peak",
            xy=(x[i], y[i]),
            xytext=(x[i] - 1.5, y[i] + 1),
            arrowprops=dict(arrowstyle="->", color="black"),
            fontsize=11)
plt.tight_layout(); plt.show()
```

**Annotation is what turns a plot into communication.** Make the
point you want the audience to take away as visible as the data.

</details>

### Stretch exercise B — ⭐⭐⭐ Bar chart with value labels and colour-coding

Plot a horizontal bar chart of monthly spend = `[1200, 1450, 920, 1810, 2300, 1750]` by month `['Jan','Feb','Mar','Apr','May','Jun']`. Highlight the *single highest* bar in red; the rest in a muted blue. Print the value of each bar at its tip.


<details>
<summary>💡 <b>Solution</b></summary>

```python
months = ["Jan","Feb","Mar","Apr","May","Jun"]
spend  = np.array([1200, 1450, 920, 1810, 2300, 1750])

colors = ["#4C72B0"] * len(spend)
colors[int(np.argmax(spend))] = "#C44E52"

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(months, spend, color=colors, edgecolor="black")
for bar, v in zip(bars, spend):
    ax.text(v + 30, bar.get_y() + bar.get_height()/2,
            f"${v:,}", va="center", fontsize=10)
ax.set_title("Monthly spend  (peak highlighted)")
ax.set_xlabel("USD")
ax.set_xlim(0, spend.max() * 1.15)
plt.tight_layout(); plt.show()
```

**"Highlight the punchline" is the most impactful chart-design trick.**
The reader's eye lands on the red bar in a second — and they
immediately know which month to talk about.

</details>

### Stretch exercise C — ⭐⭐⭐ Two panels, shared x-axis

Plot two related signals on top of each other in a single figure with two stacked subplots that share the x-axis. Upper panel: a sine wave. Lower panel: its derivative (a cosine). Label each axis and give the figure a single title.

Hint: `plt.subplots(2, 1, sharex=True)`.

In [ ]:
# Your code here  👇
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 4*np.pi, 200)

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 4*np.pi, 200)

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(9, 4))
ax1.plot(x, np.sin(x), label="sin(x)")
ax1.set_ylabel("signal")
ax1.legend(loc="upper right")

ax2.plot(x, np.cos(x), color="C1", label="d/dx sin(x) = cos(x)")
ax2.set_xlabel("x")
ax2.set_ylabel("derivative")
ax2.legend(loc="upper right")

fig.suptitle("A signal and its derivative")
fig.tight_layout()
plt.show()
```

**Reasoning.** Three matplotlib habits this exercise builds. (1) Always use the **object-oriented API** (`fig, (ax1, ax2) = plt.subplots(...)`) rather than the pyplot state machine — when you have multiple axes, `plt.plot()` becomes ambiguous about which one you're drawing on. (2) `sharex=True` keeps the x-axes aligned, which is exactly what you want when the bottom panel is a derived view of the top. (3) `fig.tight_layout()` saves you from manually fiddling with subplot spacing; together with `fig.suptitle(...)` it's the right way to give the whole figure a single title.
</details>

### Stretch exercise D — ⭐⭐⭐ Annotate the peak of a curve

Plot a noisy curve (`y = sin(x) + small noise`) and add an arrow annotation pointing at its global maximum, labelled with the (x, y) coordinates of the peak. Use `ax.annotate` with `arrowprops`.

Hint: `np.argmax` gives you the index of the maximum.

In [ ]:
# Your code here  👇
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
x = np.linspace(0, 4*np.pi, 200)
y = np.sin(x) + 0.1 * rng.standard_normal(x.shape)

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
x = np.linspace(0, 4*np.pi, 200)
y = np.sin(x) + 0.1 * rng.standard_normal(x.shape)

i = int(np.argmax(y))
peak_x, peak_y = float(x[i]), float(y[i])

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(x, y, color="C0")
ax.scatter([peak_x], [peak_y], color="red", zorder=5)
ax.annotate(
    f"peak\n(x={peak_x:.2f}, y={peak_y:.2f})",
    xy=(peak_x, peak_y),
    xytext=(peak_x + 1.5, peak_y - 0.4),
    arrowprops=dict(arrowstyle="->", color="red"),
    fontsize=10, color="red",
)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_title("Noisy sine — annotated peak")
plt.tight_layout()
plt.show()
```

**Reasoning.** Annotations are how plots tell a story without forcing the reader to squint. Three details worth absorbing. (1) `xy=...` is the point you're pointing at; `xytext=...` is where the label sits — separate them by some offset so the arrow is visible. (2) `zorder=5` pushes the red dot above the line so it isn't hidden. (3) Always compute the peak from the data (`np.argmax`) rather than hard-coding coordinates — the figure stays correct if you regenerate the data with a different random seed.
</details>

## 🎁 Bonus mini-project — Build a 2×2 AI-ops dashboard

Generate a synthetic dataset of 500 LLM calls with: `tokens_in`, `cost_usd`, `latency_ms`, `model` (one of two). Then build a 2×2 dashboard:

1. **(0,0)** Histogram of `tokens_in`.
2. **(0,1)** Scatter `cost_usd` vs `latency_ms`, coloured by `model`.
3. **(1,0)** Bar chart of mean `cost_usd` per `model`.
4. **(1,1)** Box plot of `latency_ms` per `model`.

This is structurally *the same dashboard* used in NB 24 — the only difference is the column names. Master this layout and you can produce a polished one-pager for any analysis.

In [ ]:
# Your code here  👇
rng = np.random.default_rng(0)
n = 500

tokens_in    = rng.integers(120, 2_000, n)
model        = rng.choice(["gpt-4o-mini", "claude-haiku"], n)
# Cost: roughly proportional to tokens, model-dependent
price_in_per_1k = np.where(model == "gpt-4o-mini", 0.0006, 0.0008)
cost_usd        = tokens_in / 1000 * price_in_per_1k + rng.normal(0, 0.0001, n).clip(0)
# Latency: claude-haiku is slower on average
latency_ms = np.where(model == "gpt-4o-mini",
                      rng.normal(1800, 400, n),
                      rng.normal(2300, 500, n)).clip(min=200)


<details>
<summary>💡 <b>Solution</b></summary>

```python
palette = {"gpt-4o-mini": "#4C72B0", "claude-haiku": "#DD8452"}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("AI-ops dashboard — 500-call sample", fontsize=15, fontweight="bold")

# (0, 0) Tokens histogram
axes[0, 0].hist(tokens_in, bins=25, color="#4C72B0", edgecolor="black")
axes[0, 0].set_title("Tokens per call (input)")
axes[0, 0].set_xlabel("Tokens")
axes[0, 0].set_ylabel("Number of calls")

# (0, 1) Cost vs latency scatter, coloured by model
for m in ["gpt-4o-mini", "claude-haiku"]:
    sel = model == m
    axes[0, 1].scatter(latency_ms[sel], cost_usd[sel],
                       color=palette[m], edgecolor="black", alpha=0.6, s=18,
                       label=m)
axes[0, 1].set_title("Cost vs latency, by model")
axes[0, 1].set_xlabel("Latency (ms)")
axes[0, 1].set_ylabel("Cost (USD)")
axes[0, 1].legend(fontsize=9)

# (1, 0) Mean cost per model — bar chart
models = ["gpt-4o-mini", "claude-haiku"]
means  = [cost_usd[model == m].mean() for m in models]
axes[1, 0].bar(models, means, color=[palette[m] for m in models], edgecolor="black")
for x, v in zip(models, means):
    axes[1, 0].text(x, v, f"${v:.5f}", ha="center", va="bottom", fontsize=10)
axes[1, 0].set_title("Mean cost per call")
axes[1, 0].set_ylabel("Cost (USD)")
axes[1, 0].set_ylim(0, max(means) * 1.2)
axes[1, 0].grid(axis="y", alpha=0.3)

# (1, 1) Latency boxplot per model
box_data = [latency_ms[model == m] for m in models]
bp = axes[1, 1].boxplot(box_data, tick_labels=models, patch_artist=True)
for patch, m in zip(bp["boxes"], models):
    patch.set_facecolor(palette[m]); patch.set_alpha(0.75)
axes[1, 1].set_title("Latency distribution per model")
axes[1, 1].set_ylabel("Latency (ms)")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()
```

**Why this matters.** You just produced the exact deliverable an engineering manager would paste into a model-selection slide. Two minutes to generate, far more persuasive than a table. The NB 24 capstone uses the same skeleton.
</details>

## 🧠 Key takeaways

1. Use the **Figure / Axes** (`fig, ax = plt.subplots()`) style for everything beyond a one-off.
2. Always label your axes and give the plot a title — a chart without context is a riddle.
3. Match the **chart type** to the question:
   - **line** for *trend over time*,
   - **bar** for *comparing categories*,
   - **histogram** / **box** for *distributions*,
   - **scatter** for *relationships*,
   - **heatmap** for *matrices*.
4. A 2×2 **dashboard** (`subplots(2, 2)`) is enough for most executive summaries.
5. Annotate the punchline directly on the chart — don't make the reader hunt for it.
6. Use a small, consistent **palette**.
7. Save with `plt.savefig("name.png", dpi=200, bbox_inches="tight")` for reports.

## ✅ Self-assessment

- [ ] Build a single-panel plot with title, axis labels, and a legend
- [ ] Compare multiple categories with bar / horizontal-bar charts
- [ ] Show a distribution with a histogram and a box plot
- [ ] Show a relationship with a scatter and an overlaid regression line
- [ ] Build a 2×2 subplots layout with a shared `suptitle`
- [ ] Add an annotation with an arrow to the right place on the chart
- [ ] Save a chart to PNG at publication DPI

## 🚀 Next step

Continue with **Notebook 10 — Statistics That Pay for Themselves**, where you'll learn to tell real differences from noise — the foundation for evaluating every model and A/B test that follows.